# Brent COT Mapping: ICE Europe → Marouen's cot_db

Goal: show exactly how to reconstruct the CO rows in `cot_db.csv` from ICE Europe data.

Key difference from WTI: Brent is **not** in the CFTC legacy report, so Bloomberg
uses the ICE Europe disaggregated report directly:
- Commercial = Producer/Merchant
- ManagedMoney = Managed Money + Other Reportables (approximates legacy Non-Commercial)

In [1]:
import pandas as pd

ice = pd.read_csv("../downloads/ice/ice_cot.csv", low_memory=False)
marouen = pd.read_csv("../cot_data/cot_db.csv")

print(f"ice_cot:        {len(ice)} rows")
print(f"marouen cot_db: {len(marouen)} rows")

ice_cot:        8149 rows
marouen cot_db: 19776 rows


## Step 1 — Find Brent in the ICE file

In [2]:
# Handle BOM duplicate column
ice["market"] = ice["Market_and_Exchange_Names"].fillna(
    ice.get("\ufeffMarket_and_Exchange_Names", "")
)

# What Brent markets exist?
brent = ice[ice["market"].str.contains("Brent", case=False, na=False)]
print("Brent markets in ICE file:")
for name, count in brent["market"].value_counts().items():
    print(f"  {count:>4d} rows  {name}")

# What columns do we use?
print("\nColumns used for Commercial:")
print("  Prod_Merc_Positions_Long_All")
print("  Prod_Merc_Positions_Short_All")
print("\nColumns used for ManagedMoney (= MM + Other):")
print("  M_Money_Positions_Long_All + Other_Rept_Positions_Long_All")
print("  M_Money_Positions_Short_All + Other_Rept_Positions_Short_All")

Brent markets in ICE file:
   909 rows  ICE Brent Crude Futures - ICE Futures Europe
   681 rows  ICE Brent Crude Futures and Options - ICE Futures Europe

Columns used for Commercial:
  Prod_Merc_Positions_Long_All
  Prod_Merc_Positions_Short_All

Columns used for ManagedMoney (= MM + Other):
  M_Money_Positions_Long_All + Other_Rept_Positions_Long_All
  M_Money_Positions_Short_All + Other_Rept_Positions_Short_All


## Step 2 — Filter for Combined (futures + options), fall back to FutOnly pre-2013

In [3]:
# Combined = "Futures and Options"
brent_comb = ice[ice["market"].str.contains(r"Brent.*Options", case=False, na=False)].copy()

# FutOnly = "Futures" but NOT "Futures and Options"
brent_fo = ice[ice["market"].str.contains(r"Brent.*Futures\b", case=False, na=False)].copy()
brent_fo = brent_fo[~brent_fo["market"].str.contains("Options", case=False, na=False)]

for df in [brent_comb, brent_fo]:
    df["date"] = pd.to_datetime(df["As_of_Date_Form_MM/DD/YYYY"], format="mixed")
    df.drop_duplicates(subset=["date"], keep="first", inplace=True)

print(f"Combined:  {len(brent_comb)} rows, {brent_comb['date'].min().date()} to {brent_comb['date'].max().date()}")
print(f"FutOnly:   {len(brent_fo)} rows, {brent_fo['date'].min().date()} to {brent_fo['date'].max().date()}")

Combined:  681 rows, 2013-03-12 to 2026-03-24
FutOnly:   795 rows, 2011-01-04 to 2026-03-24


In [4]:
# Merge: use Combined where available, FutOnly for earlier dates
fo_only = brent_fo[~brent_fo["date"].isin(brent_comb["date"])]
brent_all = pd.concat([brent_comb, fo_only], ignore_index=True).sort_values("date")

print(f"Combined + FutOnly fallback: {len(brent_all)} rows")
print(f"  of which {len(fo_only)} are FutOnly fallback (pre-{brent_comb['date'].min().date()})")

Combined + FutOnly fallback: 795 rows
  of which 114 are FutOnly fallback (pre-2013-03-12)


## Step 3 — Extract Commercial and ManagedMoney

In [5]:
# Coerce to numeric
cols = [
    "Prod_Merc_Positions_Long_All", "Prod_Merc_Positions_Short_All",
    "M_Money_Positions_Long_All", "M_Money_Positions_Short_All",
    "Other_Rept_Positions_Long_All", "Other_Rept_Positions_Short_All",
]
for c in cols:
    brent_all[c] = pd.to_numeric(brent_all[c], errors="coerce")

# Commercial = Producer/Merchant
# ManagedMoney = Managed Money + Other Reportables
rebuilt = pd.DataFrame({
    "date": brent_all["date"].values,
    "CommercialLong":  brent_all["Prod_Merc_Positions_Long_All"].values,
    "CommercialShort": brent_all["Prod_Merc_Positions_Short_All"].values,
    "MMLong":  brent_all["M_Money_Positions_Long_All"].values + brent_all["Other_Rept_Positions_Long_All"].values,
    "MMShort": brent_all["M_Money_Positions_Short_All"].values + brent_all["Other_Rept_Positions_Short_All"].values,
})
rebuilt["CommercialNet"] = rebuilt["CommercialLong"] - rebuilt["CommercialShort"]
rebuilt["MMNet"] = rebuilt["MMLong"] - rebuilt["MMShort"]

print(f"Rebuilt CO: {len(rebuilt)} rows")
rebuilt.head()

Rebuilt CO: 795 rows


,date,CommercialLong,CommercialShort,MMLong,MMShort,CommercialNet,MMNet
0,2011-01-04,306453,555727,145910,27169,-249274,118741
1,2011-01-11,336994,593477,151451,26985,-256483,124466
2,2011-01-18,359855,580864,150799,32149,-221009,118650
3,2011-01-25,391688,566108,136013,55276,-174420,80737
4,2011-02-01,360442,549275,141330,47016,-188833,94314


## Step 4 — Compare against Marouen's cot_db

In [6]:
marouen["tradeDate"] = pd.to_datetime(marouen["tradeDate"])
co = marouen[marouen["Name"] == "CO"].dropna(subset=["Commercial_NetPosition"]).copy()

comp = pd.merge(rebuilt, co, left_on="date", right_on="tradeDate", how="inner")
print(f"Matched rows: {len(comp)} (Marouen has {len(co)})\n")

pairs = [
    ("CommercialLong",  "CommercialLongPosition"),
    ("CommercialShort", "CommercialShortPosition"),
    ("CommercialNet",   "Commercial_NetPosition"),
    ("MMLong",          "ManagedMoney_LongPosition"),
    ("MMShort",         "ManagedMoney_ShortPosition"),
    ("MMNet",           "ManagedMoney_NetPosition"),
]

print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
print("-" * 90)
for ours, theirs in pairs:
    abs_diff = (comp[ours] - comp[theirs]).abs()
    mae = abs_diff.mean()
    denom = comp[theirs].abs().replace(0, float("nan"))
    mape = (abs_diff / denom * 100).mean()
    print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")

Matched rows: 753 (Marouen has 757)

rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.5404%    2462.51   52751
CommercialShort        CommercialShortPosition           0.4630%    2793.49   54649
CommercialNet          Commercial_NetPosition            0.3517%     426.40   11898
MMLong                 ManagedMoney_LongPosition         0.1925%     294.68   10835
MMShort                ManagedMoney_ShortPosition        0.8098%     463.44   9151
MMNet                  ManagedMoney_NetPosition          0.9091%     634.53   17081


## Step 5 — Zoom in: Combined period vs FutOnly fallback

The FutOnly fallback (pre-2013) excludes options delta — expect small diffs there.

In [7]:
cutoff = brent_comb["date"].min()
print(f"Combined data starts: {cutoff.date()}\n")

for label, mask in [("Combined (post-cutoff)", comp["date"] >= cutoff),
                     ("FutOnly fallback",       comp["date"] < cutoff)]:
    sub = comp[mask]
    if sub.empty:
        continue
    print(f"--- {label}: {len(sub)} rows ---")
    print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
    print("-" * 90)
    for ours, theirs in pairs:
        abs_diff = (sub[ours] - sub[theirs]).abs()
        mae = abs_diff.mean()
        denom = sub[theirs].abs().replace(0, float("nan"))
        mape = (abs_diff / denom * 100).mean()
        print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")
    print()

Combined data starts: 2013-03-12

--- Combined (post-cutoff): 641 rows ---
rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.0000%       0.00   0
CommercialShort        CommercialShortPosition           0.0000%       0.00   0
CommercialNet          Commercial_NetPosition            0.0000%       0.00   0
MMLong                 ManagedMoney_LongPosition         0.0000%       0.00   0
MMShort                ManagedMoney_ShortPosition        0.0000%       0.00   0
MMNet                  ManagedMoney_NetPosition          0.0000%       0.00   0

--- FutOnly fallback: 112 rows ---
rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition        